In [2]:
# Projet (DATA MINING)  "Carthage Assurances"
# Version 2 
# les membres du groupe :
# - Ibrahim BENABDALLAH
# - skandar SOUISSI 
# - Mohamed TRIKI 
# - Yassine KOUBAA



import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import classification_report, roc_curve, roc_auc_score, accuracy_score

from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.cluster import KMeans

from kneed import KneeLocator


import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df_insurance = pd.read_excel('insurance.xlsx')
df_train = pd.read_excel("insurance.xlsx", sheet_name="train")
df_test = pd.read_excel("insurance.xlsx", sheet_name="test")
df_train

In [ ]:
df_train.head()

In [ ]:
df_train.info()

In [ ]:
df_train.isnull().sum()

In [ ]:
df_train.describe()

In [ ]:
df_train = df_train.dropna(axis=0, how='all', ignore_index=True)
df_train

Visualisation des données (analyse graphique)

analyse des colonnes

In [ ]:
num_cols = df_train.select_dtypes(include=np.number).columns
cat_cols = df_train.select_dtypes(exclude=np.number).columns

In [ ]:
df_train[num_cols].hist(figsize=(10, 8), bins=15)
plt.suptitle("Distribution of Numerical Features", fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df_train[num_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
sns.boxplot(data=df_train,x="area_m2")

In [ ]:
for col in cat_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(x=df_train[col])
    plt.title(f"Countplot of {col}")
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(df_train.isnull().astype(int), yticklabels=False,cmap="Blues_r", ax=ax)
ax.set_title('Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Colonnes')
ax.set_ylabel('Lignes')
plt.tight_layout()
plt.show()

<b>START OF PREPROCESSING</b>

In [ ]:
#replacer les valeurs de window_count 
def replace(df=df_train):
    df['window_count'] = df.loc[:, 'window_count'].replace('without', 0)
    df['window_count'] = df.loc[:, 'window_count'].replace('>=10', 10)
    df['window_count'] = df.loc[:, 'window_count'].astype(int)
    return df

df_train = replace(df_train)
df_test = replace(df_test)

df_train

In [ ]:
# Geo_Code: force string then encode

df_train["Geo_Code"] = df_train["Geo_Code"].astype(str) #example: Geo_code = 2A004
unique_geo = {v: i for i, v in enumerate(sorted(df_train["Geo_Code"].unique()))}    #unique_geo = {'2A004': 0, '14079': 1, '64249': 2, ...}
df_train["Geo_Code"] = df_train["Geo_Code"].map(unique_geo).astype(int)     #map the string codes to integers
    
df_test["Geo_Code"] = df_test["Geo_Code"].astype(str)
df_test["Geo_Code_num"] = df_test["Geo_Code"].map(unique_geo)
df_test["Geo_Code_num"] = df_test["Geo_Code_num"].fillna(-1).astype(int)
df_test["Geo_Code"] = df_test["Geo_Code_num"]
df_test.drop(columns=["Geo_Code_num"], inplace=True)

df_train

In [ ]:
#deviser la colonne is_finished_and_fenced en deux colonnes
def split(df=df_train):
    df[['is_finished', 'is_fenced']] = df['is_finished_and_fenced'].str.split('/', expand=True)
    df = df.drop(columns=['is_finished_and_fenced'])
    # retour au ordre initial des colonnes (kima keno)
    df = df.iloc[:, [0,1,2,3,-1,-2, 4,5,6,7,8,9,10,11]]
    return df
    


df_train = split(df_train)
df_test = split(df_test)

df_train

In [ ]:
# replacer les valeurs V et N par 1 et 0 (kima Ordinal Encoder)
def ordinal(df=df_train):
    df['is_fenced'] = df['is_fenced'].map({'V': 1, 'N': 0})
    df['is_finished'] = df['is_finished'].map({'V': 1, 'N': 0})
    df['has_garden'] = df['has_garden'].replace({np.nan: 0, 'V': 1, 'O': 0})
    df['claim']= df['claim'].map({'oui':1, 'non':0})
    return df

df_train = ordinal(df_train)
df_test = ordinal(df_test)

df_train

In [ ]:
#one hot encoding pour la colonne locality
locality_onehot = OneHotEncoder(sparse_output=False, categories=[["U", "R"]], handle_unknown='ignore')
x = locality_onehot.fit_transform(df_train.loc[:,["locality"]])
df_x = pd.DataFrame(x, columns=["urbaine", "rurale"])
df_train = pd.concat([df_x, df_train], axis=1)
df_train = df_train.drop(columns=["locality"])

x = locality_onehot.transform(df_test.loc[:,["locality"]])
df_x = pd.DataFrame(x, columns=["urbaine", "rurale"])
df_test = pd.concat([df_x, df_test], axis=1)
df_test = df_test.drop(columns=["locality"])

df_train

In [ ]:
# one hot encoding pour la colonne structure_type
struct_onehot = OneHotEncoder(sparse_output=False, categories=[["Wood-framed", "Non-combustible", "Ordinary", "Fire-resistive"]], handle_unknown='ignore')
x = struct_onehot.fit_transform(df_train.loc[:,["structure_type"]])
df_x = pd.DataFrame(x, columns=["Wood-framed", "Non-combustible", "Ordinary", "Fire-resistive"])
df_train = pd.concat([df_x, df_train], axis=1)
df_train = df_train.drop(columns=["structure_type"])

x = struct_onehot.transform(df_test.loc[:,["structure_type"]])
df_x = pd.DataFrame(x, columns=["Wood-framed", "Non-combustible", "Ordinary", "Fire-resistive"])
df_test = pd.concat([df_x, df_test], axis=1)
df_test = df_test.drop(columns=["structure_type"])

df_train

In [ ]:
#valeurs aberrantes de area_m2

Q1 = df_train['area_m2'].quantile(0.25)
Q3 = df_train['area_m2'].quantile(0.75)
IQR = Q3 - Q1
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR
df_train = df_train[(df_train['area_m2'] >= borne_inf) & (df_train['area_m2'] <= borne_sup)]

df_train['area_m2'] = df_train['area_m2'].fillna(df_train['area_m2'].median())

In [ ]:
# fill NaN

def fill_nan(df=df_train):
    df["area_m2"] = df["area_m2"].fillna(df["area_m2"].median())
    df["risk_score"] = df["risk_score"].fillna(0)
    return df

df_train = fill_nan(df_train)
df_test = fill_nan(df_test)

In [ ]:
# convertir les colonnes en int

def convert_int(df=df_train):
    df = df.astype({'is_residential': 'int64'})
    df = df.astype({'coverage_period': 'int64'})
    df = df.astype({'year': 'int64'})
    
    df = df.astype({'urbaine': 'int64'})
    df = df.astype({'rurale': 'int64'})

    df = df.astype({'Fire-resistive': 'int64'})
    df = df.astype({'Non-combustible': 'int64'})
    df = df.astype({'Ordinary': 'int64'})
    df = df.astype({'Wood-framed': 'int64'})
    
    return df
    
df_train = convert_int(df_train)
df_test = convert_int(df_test)

df_train

In [ ]:
sns.boxplot(data=df_train,x="area_m2")

In [ ]:
feature_cols = [
    "year",
    "coverage_period",
    "is_residential",
    "has_garden",
    "urbaine",
    "rurale",
    "area_m2",
    "window_count",
    "risk_score",
    "Geo_Code",
    "Wood-framed",
    "Non-combustible",
    "Ordinary",
    "Fire-resistive",
    "is_fenced",
    "is_finished"
]

X = df_train[feature_cols].copy()
y = df_train["claim"].copy()

X_test = df_test[feature_cols].copy()
y_test = df_test["claim"].copy()

smote = SMOTE(random_state=42)
X_train_smote, Y_train_smote = smote.fit_resample(X, y)

# sécurité : aucune colonne texte
for col in X.columns:
    if X[col].dtype == "object":
        print("⚠️ Still text:", col)
        raise ValueError("STOP: non-encoded feature found.")
    
# =========================
# Preprocessor  
# =========================
#ta74ir pipeline

numeric_features = [
    "year", "coverage_period", "area_m2",
    "window_count", "risk_score", "Geo_Code"
]

categorical_features = [c for c in feature_cols if c not in numeric_features]

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=Y_train_smote)
plt.title(f"Countplot of claim")
plt.show()

In [ ]:
# use the numeric columns defined earlier
num_features = df_train.loc[:, num_cols]

# Standardize the numerical features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(num_features.astype(np.float64))

# Calculate inertia for different numbers of clusters
inertias = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_features)
    inertias.append(kmeans.inertia_)

# Plot the Elbow graph
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bx-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k (for StratifiedKFold n_splits reference)')
plt.grid()
plt.show()

try:
    kl = KneeLocator(K_range, inertias, curve='convex', direction='decreasing')
    elbow_point = kl.elbow
    print(f"Elbow point detected at k = {elbow_point}")
    print(f"This suggests StratifiedKFold with n_splits around {elbow_point} might be reasonable.")
except ImportError:
    print("For automatic elbow detection, install: pip install kneed")
    print("Otherwise, visually inspect the plot for the 'elbow'.")

<b>Apprentissage automatique</b>

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", "passthrough", categorical_features),
    ]
)

to_float = FunctionTransformer(lambda x: x.astype(np.float32))

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)

#ta74ir stacking
stacking_base_estimators = [
    ("lr", LogisticRegression(max_iter=2000)),
    ("dt", DecisionTreeClassifier(random_state=0)),
    ("rf", RandomForestClassifier(n_estimators=150, random_state=0)),
    ("svm", SVC(kernel="rbf", probability=True, random_state=0))
]

meta_classifier = LogisticRegression(max_iter=2000)

stacking_clf = StackingClassifier(
    estimators=stacking_base_estimators,
    final_estimator=meta_classifier,
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)

#ta74ir voting
voting_estimators = [
    ("lr", LogisticRegression(max_iter=2000)),
    ("rf", RandomForestClassifier(n_estimators=150, random_state=0)),
    ("gb", GradientBoostingClassifier(random_state=0)),
    ("svm", SVC(kernel="rbf", probability=True, random_state=0))
]

voting_clf = VotingClassifier(
    estimators=voting_estimators,
    voting="soft",   # needed for ROC-AUC
    n_jobs=-1
)

In [ ]:
# =========================
# 5. Cross-validation
# =========================
def cross_validate_model(X, Y, models):
    results = []

    for name, clf in models.items():
        print(f"\n=== Model : {name} ===")
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("to_float", to_float),
            ("classifier", clf)
        ])

        scores = cross_validate(
            pipe, X, Y,
            cv=cv,
            scoring=["accuracy", "roc_auc"],
            return_train_score=False
        )

        print("Accuracy mean :", scores["test_accuracy"].mean())
        print("ROC_AUC mean  :", scores["test_roc_auc"].mean())

        results.append({
            "model": name,
            "accuracy_mean": scores["test_accuracy"].mean(),
            "roc_auc_mean": scores["test_roc_auc"].mean(),
        })

    print("\n===== SUMMARY =====")
    print(pd.DataFrame(results))

    plt.figure()

    # =========================
    # 7. Entraînement complet + rapports & ROC
    # =========================
    for name, clf in models.items():
        print(f"\n===== {name} =====")
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("to_float", to_float),
            ("classifier", clf)
        ])

        # entraînement sur tout le train
        pipe.fit(X, Y)

        # prédictions sur test
        y_pred = pipe.predict(X_test)

        # proba / score pour ROC
        if hasattr(pipe.named_steps["classifier"], "predict_proba"):
            y_proba = pipe.predict_proba(X_test)[:, 1]
        else:
            y_proba = pipe.decision_function(X_test)

        # 1) Rapport de classification
        print("=== Classification report ===")
        print(classification_report(y_test, y_pred))

        # 2) Courbe ROC + AUC
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc_roc = roc_auc_score(y_test, y_proba)

        
        plt.plot(fpr, tpr, label=f"ROC {name} (AUC = {auc_roc:.3f})")
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.legend()
        
    plt.show()

Default version

In [ ]:
# # =========================
# # Models
# # =========================
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "DecisionTree": DecisionTreeClassifier(random_state=0),
    "RandomForest": RandomForestClassifier(n_estimators=150, random_state=0),
    "GradientBoosting": GradientBoostingClassifier(random_state=0),
    "SVM": SVC(kernel="rbf", probability=True, random_state=0),
    "MLP": MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=0),
}

cross_validate_model(X, y, models)
cross_validate_model(X_train_smote, Y_train_smote, models)

In [ ]:
models = {
    "voting_clf": voting_clf,
    "stacking_clf": stacking_clf,
    "Bagging_DT": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=0),
        n_estimators=100,
        max_samples=0.8,
        max_features=1.0,
        bootstrap=True,
        random_state=0,
        n_jobs=-1
    )
}

cross_validate_model(X, y, models)
cross_validate_model(X_train_smote, Y_train_smote, models)

Stacking version

In [ ]:


# models = {
#     "LogisticRegression": LogisticRegression(max_iter=2000),
#     "DecisionTree": DecisionTreeClassifier(random_state=0),
#     "RandomForest": RandomForestClassifier(n_estimators=150, random_state=0),
#     "GradientBoosting": GradientBoostingClassifier(random_state=0),
#     "SVM": SVC(kernel="rbf", probability=True, random_state=0),
#     "MLP": MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=0),
#     "Stacking": stacking_clf
# }

# cross_validate_model(X, y, models)
# cross_validate_model(X_train_smote, Y_train_smote, models)

Bagging version

In [ ]:
# models = {
#     "LogisticRegression": LogisticRegression(max_iter=2000),
#     "DecisionTree": DecisionTreeClassifier(random_state=0),
#     "RandomForest": RandomForestClassifier(n_estimators=150, random_state=0),
#     "GradientBoosting": GradientBoostingClassifier(random_state=0),
#     "SVM": SVC(kernel="rbf", probability=True, random_state=0),
#     "MLP": MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=0),
#     "Bagging_DT": BaggingClassifier(
#         estimator=DecisionTreeClassifier(random_state=0),
#         n_estimators=100,
#         max_samples=0.8,
#         max_features=1.0,
#         bootstrap=True,
#         random_state=0,
#         n_jobs=-1
#     ),
# }

# cross_validate_model(X, y, models)
# cross_validate_model(X_train_smote, Y_train_smote, models)

Voting version

In [ ]:


# models = {
#     "LogisticRegression": LogisticRegression(max_iter=2000),
#     "DecisionTree": DecisionTreeClassifier(random_state=0),
#     "RandomForest": RandomForestClassifier(n_estimators=150, random_state=0),
#     "GradientBoosting": GradientBoostingClassifier(random_state=0),
#     "SVM": SVC(kernel="rbf", probability=True, random_state=0),
#     "MLP": MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=0),
    
#     "Voting": voting_clf
# }

# cross_validate_model(X, y, models)
# cross_validate_model(X_train_smote, Y_train_smote, models)

In [ ]:
# LR = LogisticRegression(max_iter=2000)

# result = []

# print(f"\n=== Model : LogisticRegression ===")
# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", LR)
#     ])

# scores = cross_validate(
#         pipe, X, y,
#         cv=cv,
#         scoring=["accuracy", "roc_auc"],
#         return_train_score=False
#     )

# print("Accuracy mean :", scores["test_accuracy"].mean())
# print("ROC_AUC mean  :", scores["test_roc_auc"].mean())


# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", LR)
# ])

# # entraînement sur tout le train
# pipe.fit(X, y)

# # prédictions sur test
# y_pred = pipe.predict(X_test)

# # proba / score pour ROC
# if hasattr(pipe.named_steps["classifier"], "predict_proba"):
#     y_proba = pipe.predict_proba(X_test)[:, 1]
# else:
#     y_proba = pipe.decision_function(X_test)

# # 1) Rapport de classification
# print("=== Classification report ===")
# print(classification_report(y_test, y_pred))

# # 2) Courbe ROC + AUC
# fpr, tpr, _ = roc_curve(y_test, y_proba)
# auc_roc = roc_auc_score(y_test, y_proba)

# plt.figure()
# plt.plot(fpr, tpr, label=f"ROC (AUC = {auc_roc:.3f})")
# plt.plot([0, 1], [0, 1], 'k--')
# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.title(f"ROC - LogisticRegression")
# plt.legend()
# plt.show()

In [ ]:
# DT = DecisionTreeClassifier(random_state=0)

# result = []

# print(f"\n=== Model : DecisionTreeClassifier ===")
# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", DT)
#     ])

# scores = cross_validate(
#         pipe, X, y,
#         cv=cv,
#         scoring=["accuracy", "roc_auc"],
#         return_train_score=False
#     )

# print("Accuracy mean :", scores["test_accuracy"].mean())
# print("ROC_AUC mean  :", scores["test_roc_auc"].mean())

# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", DT)
# ])

# # entraînement sur tout le train
# pipe.fit(X, y)

# # prédictions sur test
# y_pred = pipe.predict(X_test)

# # proba / score pour ROC
# if hasattr(pipe.named_steps["classifier"], "predict_proba"):
#     y_proba = pipe.predict_proba(X_test)[:, 1]
# else:
#     y_proba = pipe.decision_function(X_test)

# # 1) Rapport de classification
# print("=== Classification report ===")
# print(classification_report(y_test, y_pred))

# # 2) Courbe ROC + AUC
# fpr, tpr, _ = roc_curve(y_test, y_proba)
# auc_roc = roc_auc_score(y_test, y_proba)

# plt.figure()
# plt.plot(fpr, tpr, label=f"ROC (AUC = {auc_roc:.3f})")
# plt.plot([0, 1], [0, 1], 'k--')
# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.title(f"ROC - DecisionTreeClassifier")
# plt.legend()
# plt.show()

In [ ]:
# RFS = RandomForestClassifier(n_estimators=150, random_state=0)

# result = []

# print(f"\n=== Model : RandomForestClassifier ===")
# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", RFS)
#     ])

# scores = cross_validate(
#         pipe, X, y,
#         cv=cv,
#         scoring=["accuracy", "roc_auc"],
#         return_train_score=False
#     )

# print("Accuracy mean :", scores["test_accuracy"].mean())
# print("ROC_AUC mean  :", scores["test_roc_auc"].mean())

# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", RFS)
# ])

# # entraînement sur tout le train
# pipe.fit(X, y)

# # prédictions sur test
# y_pred = pipe.predict(X_test)

# # proba / score pour ROC
# if hasattr(pipe.named_steps["classifier"], "predict_proba"):
#     y_proba = pipe.predict_proba(X_test)[:, 1]
# else:
#     y_proba = pipe.decision_function(X_test)

# # 1) Rapport de classification
# print("=== Classification report ===")
# print(classification_report(y_test, y_pred))

# # 2) Courbe ROC + AUC
# fpr, tpr, _ = roc_curve(y_test, y_proba)
# auc_roc = roc_auc_score(y_test, y_proba)

# plt.figure()
# plt.plot(fpr, tpr, label=f"ROC (AUC = {auc_roc:.3f})")
# plt.plot([0, 1], [0, 1], 'k--')
# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.title(f"ROC - DecisionTreeClassifier")
# plt.legend()
# plt.show()

In [ ]:
# GBC = GradientBoostingClassifier(random_state=0)

# result = []

# print(f"\n=== Model : GradientBoostingClassifier ===")
# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", GBC)
#     ])

# scores = cross_validate(
#         pipe, X, y,
#         cv=cv,
#         scoring=["accuracy", "roc_auc"],
#         return_train_score=False
#     )

# print("Accuracy mean :", scores["test_accuracy"].mean())
# print("ROC_AUC mean  :", scores["test_roc_auc"].mean())

# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", GBC)
# ])

# # entraînement sur tout le train
# pipe.fit(X, y)

# # prédictions sur test
# y_pred = pipe.predict(X_test)

# # proba / score pour ROC
# if hasattr(pipe.named_steps["classifier"], "predict_proba"):
#     y_proba = pipe.predict_proba(X_test)[:, 1]
# else:
#     y_proba = pipe.decision_function(X_test)

# # 1) Rapport de classification
# print("=== Classification report ===")
# print(classification_report(y_test, y_pred))

# # 2) Courbe ROC + AUC
# fpr, tpr, _ = roc_curve(y_test, y_proba)
# auc_roc = roc_auc_score(y_test, y_proba)

# plt.figure()
# plt.plot(fpr, tpr, label=f"ROC (AUC = {auc_roc:.3f})")
# plt.plot([0, 1], [0, 1], 'k--')
# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.title(f"ROC - GradientBoostingClassifier")
# plt.legend()
# plt.show()

In [ ]:
# SVM = SVC(kernel="rbf", probability=True, random_state=0)

# result = []

# print(f"\n=== Model : SVC ===")
# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", SVM)
#     ])

# scores = cross_validate(
#         pipe, X, y,
#         cv=cv,
#         scoring=["accuracy", "roc_auc"],
#         return_train_score=False
#     )

# print("Accuracy mean :", scores["test_accuracy"].mean())
# print("ROC_AUC mean  :", scores["test_roc_auc"].mean())

# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", SVM)
# ])

# # entraînement sur tout le train
# pipe.fit(X, y)

# # prédictions sur test
# y_pred = pipe.predict(X_test)

# # proba / score pour ROC
# if hasattr(pipe.named_steps["classifier"], "predict_proba"):
#     y_proba = pipe.predict_proba(X_test)[:, 1]
# else:
#     y_proba = pipe.decision_function(X_test)

# # 1) Rapport de classification
# print("=== Classification report ===")
# print(classification_report(y_test, y_pred))

# # 2) Courbe ROC + AUC
# fpr, tpr, _ = roc_curve(y_test, y_proba)
# auc_roc = roc_auc_score(y_test, y_proba)

# plt.figure()
# plt.plot(fpr, tpr, label=f"ROC (AUC = {auc_roc:.3f})")
# plt.plot([0, 1], [0, 1], 'k--')
# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.title(f"ROC - SVC")
# plt.legend()
# plt.show()

In [ ]:
# MLP = MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=0)

# results = []

# print(f"\n=== Model : MLPClassifier ===")
# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", MLP)
#     ])

# scores = cross_validate(
#         pipe, X, y,
#         cv=cv,
#         scoring=["accuracy", "roc_auc"],
#         return_train_score=False
#     )

# print("Accuracy mean :", scores["test_accuracy"].mean())
# print("ROC_AUC mean  :", scores["test_roc_auc"].mean())

# pipe = Pipeline([
#     ("preprocessor", preprocessor),
#     ("to_float", to_float),
#     ("classifier", MLP)
# ])

# # entraînement sur tout le train
# pipe.fit(X, y)

# # prédictions sur test
# y_pred = pipe.predict(X_test)

# # proba / score pour ROC
# if hasattr(pipe.named_steps["classifier"], "predict_proba"):
#     y_proba = pipe.predict_proba(X_test)[:, 1]
# else:
#     y_proba = pipe.decision_function(X_test)

# # 1) Rapport de classification
# print("=== Classification report ===")
# print(classification_report(y_test, y_pred))

# # 2) Courbe ROC + AUC
# fpr, tpr, _ = roc_curve(y_test, y_proba)
# auc_roc = roc_auc_score(y_test, y_proba)

# plt.figure()
# plt.plot(fpr, tpr, label=f"ROC (AUC = {auc_roc:.3f})")
# plt.plot([0, 1], [0, 1], 'k--')
# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.title(f"ROC - MLPClassifier")
# plt.legend()
# plt.show()